# La France est-elle en retard en IA, ou simplement plus transparente ?

*Projet Python pour la data science — ENSAE 2A, 2026-2027*

Ce notebook est le **rapport final** : il enchaîne collecte, nettoyage, analyse et conclusions, et s'exécute de bout en bout. Le détail de chaque analyse (tests de robustesse, graphiques complémentaires) est dans les notebooks d'annexe du dossier `notebooks/`.

## Introduction

> *Premier jet : chaque paragraphe d'interprétation est à relire, vérifier sur les sorties et reformuler avant le rendu.*

Depuis la sortie de ChatGPT fin 2022, l'IA est devenue un enjeu de souveraineté. La France y a mis des moyens importants et compte avec Mistral AI l'un des rares développeurs européens de modèles de langage. Dans le même temps, l'Union européenne a adopté l'AI Act, dont les obligations pour les modèles d'IA à usage général s'appliquent depuis le 2 août 2025, accompagnées d'un code de bonnes pratiques que chaque développeur peut signer. Côté usages, 18 % des entreprises implantées en France déclarent utiliser au moins une technologie d'IA en 2025 ([Insee](https://www.insee.fr/fr/statistiques/9025878)).

On entend souvent que la France et l'Europe sont « en retard ». Mais mesurer ce retard suppose de connaître les modèles des leaders, or ceux-ci publient de moins en moins d'informations. D'où notre question : **la France est-elle en retard en IA, ou simplement plus transparente ?** Autrement dit, à quel maillon la France décroche-t-elle, et l'opacité des leaders fausse-t-elle la comparaison ?

## Démarche

On mesure la place de la France **maillon par maillon** — recherche, modèles, diffusion — puis l'**opacité** des développeurs, avant de se demander si cette opacité biaise la mesure du retard.

Hypothèses posées avant l'analyse :

1. **H1** — La France tient son rang en recherche mais décroche quand on passe des articles aux modèles.
2. **H2** — Sa présence parmi les modèles repose sur très peu d'acteurs.
3. **H3** — Les développeurs publient de moins en moins d'informations sur leurs modèles depuis 2020.
4. **H4** — Les modèles aux poids ouverts sont mieux documentés que les modèles fermés.
5. **H5** — Les signataires du code de bonnes pratiques européen (AI Act) sont devenus plus transparents.

Pays comparés : États-Unis, Chine, Royaume-Uni, France, Allemagne, Canada.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd() / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import spacy

from config import FIGURES
from style import COULEURS_PAYS, appliquer_style
from nettoyage import (charger_dernier, preparer_epoch, eclater_pays,
                       annees_de_retard, INFOS_PUBLIABLES)
from texte import nettoyer, frequences, termes_surrepresentes, nuage
from graphiques import courbes_par_pays, profil_maillons, barres_divergentes, barres_par_pays
import analyses

appliquer_style()
FIGURES.mkdir(exist_ok=True)
pd.set_option("display.precision", 2)

## 1. Les données

### Collecte

Six sources, chacune collectée par un script de `src/` : Epoch AI (CSV), OpenAlex (API), Hugging Face (API), le Foundation Model Transparency Index de Stanford (CSV sur GitHub), la liste des signataires du code européen (web scraping) et la Banque mondiale (API).

Chaque collecte enregistre une copie datée dans `data/raw/` et une copie figée en Parquet sur le stockage MinIO du SSP Cloud. Les API changent : le rapport relit par défaut ces copies figées. Passer `COLLECTER = True` relance toute la collecte (une dizaine de minutes ; une clé OpenAlex gratuite dans la variable d'environnement `OPENALEX_API_KEY` est conseillée).

In [ ]:
COLLECTER = False

if COLLECTER:
    from pipeline import collecter_tout
    collecter_tout()

In [ ]:
epoch = preparer_epoch(charger_dernier("epoch_notable_models"))
epoch_pays = eclater_pays(epoch)
publications = charger_dernier("openalex_publications")
resumes = charger_dernier("openalex_resumes")
hf = charger_dernier("hf_models")
fmti = charger_dernier("fmti_scores")
signataires = charger_dernier("signataires_code")
pays_info = charger_dernier("banque_mondiale")

pd.DataFrame({
    "lignes": [len(epoch), len(publications), len(resumes), len(hf), fmti["developpeur"].nunique(), len(signataires), len(pays_info)],
    "unité": ["modèles", "pays × années", "résumés d'articles", "modèles", "développeurs notés", "signataires", "pays × années"],
    "période": [f"{epoch['annee'].min()}-{epoch['annee'].max()}",
                f"{publications['annee'].min()}-{publications['annee'].max()}",
                f"{resumes['annee'].min()}-{resumes['annee'].max()}",
                f"{hf['createdAt'].str[:4].min()}-{hf['createdAt'].str[:4].max()}",
                ", ".join(sorted(fmti["edition"].unique())), "liste à date",
                f"{pays_info['annee'].min()}-{pays_info['annee'].max()}"],
}, index=["Epoch AI", "OpenAlex (comptages)", "OpenAlex (résumés)", "Hugging Face",
         "Stanford FMTI", "Code européen", "Banque mondiale"])

### Choix de nettoyage

- **Pays d'un modèle** : Epoch donne le pays de chaque organisation. Un modèle développé par plusieurs organisations est réparti entre elles (**comptage fractionnaire**) ; le comptage entier sert de test de robustesse (partie 2.3).
- **Ouverture** : lue dans la colonne « Model accessibility » (poids ouverts ou non) ; une valeur vide reste inconnue plutôt que comptée comme « fermé ».
- **Opacité** : pour chaque modèle, cinq indicateurs « information renseignée ou non » (puissance de calcul, paramètres, taille des données, coût, matériel), et la **source** de la puissance de calcul d'après les notes d'Epoch (calculée à partir d'informations publiées, ou devinée).
- **Publications** : OpenAlex, sous-domaine « Artificial Intelligence », articles, prépublications **et actes de conférence** (premier canal de publication en IA).
- **Rapprochement des organisations** entre Epoch, Stanford, Hugging Face et la liste de la Commission : table de correspondance explicite (`cle_organisation`).
- **Corpus NLP** : articles d'informatique quantique retirés (classés à tort en « IA » par OpenAlex).

## 2. Où la France décroche-t-elle ?

### 2.1 La France maillon par maillon

Part de chaque pays parmi les six comparés, sur 2024-2025 : articles d'IA, articles dans le 1 % les plus cités, modèles notables, modèles de pointe, téléchargements des modèles ouverts sur Hugging Face.

In [ ]:
chaine = analyses.chaine_par_periode(publications, epoch_pays)
profil = analyses.profil_recent(chaine, hf)
profil_maillons(profil, "Part de chaque pays, de la recherche à la diffusion (2024-2025)",
                fichier=FIGURES / "rapport_1_profil.png")
plt.show()
profil.style.format("{:.1%}")

**Interprétation.** Le profil de la France confirme **H1** : sa part baisse à mesure qu'on avance dans la chaîne. Elle pèse 4,3 % des articles d'IA des six pays et 2,3 % des articles les plus cités, 2,6 % des modèles notables, et aucun modèle de pointe sur 2024-2025. Le volume d'articles est trompeur : rapporté à la population, la France publie 171 articles d'IA par million d'habitants sur 2022-2024, au niveau des États-Unis (188) et de l'Allemagne (190), mais derrière le Royaume-Uni (277) et le Canada (254). Le décrochage net se situe donc entre la recherche et les modèles de pointe, que les États-Unis concentrent à 94 %. La diffusion ne rattrape pas l'écart : 1,9 % des téléchargements sur Hugging Face, quand la Chine (50 %) dépasse désormais les États-Unis (46 %) grâce à ses modèles ouverts.

### 2.2 De la recherche aux modèles

In [ ]:
transformation = chaine.pivot(index="pays", columns="periode", values="modeles_pour_1000_articles").reindex(analyses.PAYS)
transformation.round(2)

**Interprétation.** Sur 2024-2025, la France produit 0,6 modèle notable pour 1 000 articles d'IA, contre 2,1 aux États-Unis. Le ratio reste stable depuis 2018 pour la France (0,66 puis 0,74), alors que la Chine le triple (0,23 → 0,70) : elle transforme de mieux en mieux sa recherche en modèles. La chute du Royaume-Uni (1,34 → 0,13) est en partie un artefact : depuis la fusion de 2023, Epoch attribue « Google DeepMind » aux États-Unis. Les ratios mélangent un comptage entier (articles) et fractionnaire (modèles) : ils se lisent en comparaison et en tendance, pas en valeur absolue.

### 2.3 Une présence qui tient à peu d'acteurs

In [ ]:
fr = epoch_pays.loc[epoch_pays["pays"] == "France"]
par_organisation = fr.groupby("Organization")["poids"].sum().sort_values(ascending=False)
depuis_2024 = fr.loc[fr["annee"] >= 2024]
print(f"Modèles français depuis 2024 : {depuis_2024['poids'].sum():.1f}, "
      f"dont Mistral AI : {depuis_2024.loc[depuis_2024['Organization'].str.contains('Mistral', na=False), 'poids'].sum():.1f}")

comptages = pd.DataFrame({
    "fractionnaire": epoch_pays.groupby("pays")["poids"].sum(),
    "entier": eclater_pays(epoch, "entier").groupby("pays")["poids"].sum(),
}).reindex(analyses.PAYS)
comptages["entier / fractionnaire"] = comptages["entier"] / comptages["fractionnaire"]
display(par_organisation.head(5).round(2))
comptages.round(2)

**Interprétation.** **H2** est confirmée : depuis 2024, les sept modèles notables français sont tous de Mistral AI. Sur toute la période, la présence française passe aussi par les collaborations : en comptage entier, la France compte 52 modèles au lieu de 28,7 (×1,8), contre ×1,1 pour les États-Unis. Une bonne part vient du laboratoire parisien de Facebook AI Research, co-attribué aux États-Unis et à la France. La position française est donc fragile : elle repose sur une start-up et sur le laboratoire d'une entreprise américaine.

### 2.4 Le retard en années de puissance de calcul

La puissance de calcul des modèles (en FLOP) double à un rythme régulier : on l'estime par une régression log-linéaire sur la date. Pour chaque pays, on cherche ensuite en quelle année les États-Unis avaient atteint son record de puissance : l'écart donne un retard en années.

In [ ]:
display(analyses.temps_de_doublement(epoch_pays))
retard = annees_de_retard(epoch_pays).pivot(index="annee", columns="pays", values="retard_annees")
retard = retard.loc[retard.index >= 2018, [p for p in analyses.PAYS if p != "États-Unis"]]
courbes_par_pays(retard, "Retard sur les États-Unis, en années de puissance de calcul",
                 "Années de retard", fichier=FIGURES / "rapport_2_retard.png")
plt.show()

**Interprétation.** La puissance de calcul des modèles notables double environ tous les 5,4 mois (IC 95 % : 5,0-5,7), un peu plus vite en Chine (4,9) qu'aux États-Unis (5,6). Dans cette course, la France reste de 1 à 4 ans derrière les États-Unis depuis 2018 : en 2026, son record correspond au niveau américain de 2023 (3 ans de retard). Le retard ne se résorbe pas. La mesure reste fragile : elle repose sur les seuls modèles dont la puissance de calcul est connue, et un seul modèle peut faire varier le retard d'une année.

## 3. Les développeurs deviennent-ils opaques ?

### 3.1 Une information qui disparaît

In [ ]:
evolution = epoch.loc[epoch["annee"] >= 2015].groupby("annee")[list(INFOS_PUBLIABLES.values())].mean()
noms = {"publie_calcul": "Puissance de calcul", "publie_parametres": "Paramètres",
        "publie_donnees": "Taille des données", "publie_materiel": "Matériel", "publie_cout": "Coût"}
fig, ax = plt.subplots()
for (col, nom), c in zip(noms.items(), ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]):
    ax.plot(evolution.index, evolution[col], color=c, label=nom, linewidth=3 if col == "publie_calcul" else 1.6)
ax.axvline(2025.6, color="#8a8984", linestyle="--", linewidth=1)
ax.text(2025.5, 0.97, "AI Act", ha="right", fontsize=9, color="#52514e")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_ylim(0, 1)
ax.set_title("Part des modèles notables dont l'information est connue")
ax.legend(ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.08), fontsize=9)
fig.tight_layout(); fig.savefig(FIGURES / "rapport_3_opacite.png"); plt.show()
evolution.loc[[2018, 2021, 2024, 2025, 2026]].style.format("{:.0%}")

**Interprétation.** **H3** est confirmée. La part des modèles notables dont la puissance de calcul est connue culmine à 78 % en 2021, puis tombe à 46 % en 2024 et 25 % en 2026. La taille des données d'entraînement suit la même pente, et le coût n'est presque plus jamais connu (1 % en 2026). Le retournement coïncide avec la commercialisation des grands modèles à partir de 2022. Les deux dernières années sont en partie mécaniques : Epoch complète ses fiches avec retard.

### 3.2 Information publiée ou devinée ?

In [ ]:
sources = (pd.crosstab(epoch.loc[epoch["annee"] >= 2018, "annee"],
                       epoch.loc[epoch["annee"] >= 2018, "source_calcul"], normalize="index")
           .reindex(columns=["calculée depuis des infos publiées", "estimation indirecte", "non renseignée"], fill_value=0))
sources.style.format("{:.0%}")

**Interprétation.** Le recul ne vient pas d'Epoch qui devinerait moins : les estimations indirectes restent marginales (5 à 16 % des modèles selon les années depuis 2018). Ce qui baisse, c'est la part des modèles dont la puissance de calcul peut être **calculée à partir d'informations publiées** par le développeur : 67 % en 2021, 20 % en 2026. C'est donc bien la divulgation qui recule.

### 3.3 Qui publie ?

Régression logistique : probabilité que la puissance de calcul d'un modèle soit connue, selon l'année, l'ouverture des poids, le caractère de pointe et le pays (référence : États-Unis), modèles publiés depuis 2018. Un rapport de chances supérieur à 1 augmente la probabilité de publication.

In [ ]:
rapports, modele_logit = analyses.logit_publication(epoch_pays)
print(f"n = {int(modele_logit.nobs)}, pseudo-R² = {modele_logit.prsquared:.3f}")
rapports

**Interprétation.** Toutes choses égales par ailleurs, les chances que la puissance de calcul d'un modèle soit connue sont multipliées par 0,75 chaque année depuis 2018 (IC : 0,68-0,83) : elles baissent d'un quart par an. Un modèle aux poids ouverts a 3,8 fois plus de chances d'être documenté, ce qui va dans le sens de **H4**. Les modèles de pointe sont mieux renseignés (×7,4), sans doute parce qu'Epoch les étudie en priorité. Les écarts entre pays ne sont pas significatifs pour la France (p = 0,33). Cette régression décrit des corrélations : elle ne dit pas pourquoi un développeur publie.

### 3.4 Deux mesures de la transparence qui se contredisent

Le score de Stanford note ce que chaque développeur **divulgue** (une centaine d'indicateurs) ; Epoch indique ce qui est **connu** d'un modèle. On les compare pour les développeurs présents dans les deux sources, par une corrélation de rang (Spearman), adaptée au petit nombre de développeurs.

In [ ]:
score = analyses.score_stanford(fmti, signataires)
comparaison, rho = analyses.comparer_mesures(score, epoch)
print({k: round(v, 2) for k, v in rho.items()})
comparaison.round(2)

**Interprétation.** Les deux mesures se contredisent : la corrélation de rang entre le score de Stanford et la mesure tirée d'Epoch est de −0,26, et de −0,59 avec la mesure stricte. DeepSeek, Meta et Alibaba sont bien renseignés dans Epoch mais mal notés par Stanford ; Anthropic et OpenAI, à l'inverse. Les développeurs ouverts publient des fiches techniques détaillées (taille, données d'entraînement), mais restent opaques sur le reste : sources des données, conditions de travail, usages, gouvernance. **H4 n'est donc vraie que pour la transparence technique.** Mistral, seul acteur français noté, est l'un des développeurs les moins transparents selon Stanford en 2025 (18 %, contre 55 % en 2024, avec une grille plus exigeante en 2025).

### 3.5 L'AI Act a-t-il changé quelque chose ?

Comparaison **descriptive** des modèles publiés depuis janvier 2024, avant et après l'entrée en application des obligations de l'AI Act (2 août 2025), selon que le développeur a signé le code de bonnes pratiques européen.

In [ ]:
analyses.avant_apres_ai_act(epoch, signataires).style.format(
    {"modeles": "{:.0f}", "publie_calcul": "{:.0%}", "publie_donnees": "{:.0%}"})

**Interprétation.** **H5 n'est pas confirmée.** Les signataires (OpenAI, Anthropic, Google, Mistral…) publient beaucoup moins que les non-signataires, avant comme après août 2025 : 11 % de leurs modèles récents ont une puissance de calcul connue, contre 40 %. Les deux groupes reculent après l'AI Act. Cette comparaison n'est pas causale : peu de modèles après août 2025, retard de mise à jour d'Epoch, et des signataires qui sont surtout de grands laboratoires fermés. Le résumé des données d'entraînement exigé par l'AI Act porte sur leurs sources, pas sur leur taille : il n'apparaît pas dans Epoch.

### 3.6 L'opacité fausse-t-elle la mesure du retard ?

C'est le lien entre les deux parties. Le retard de la partie 2.4 se mesure sur les seuls modèles dont la puissance de calcul est connue. Si les modèles qui manquent viennent surtout des leaders, la comparaison est biaisée. On regarde donc, pour les modèles publiés depuis 2024, quelle part a une puissance de calcul connue, par pays puis pour les quatre grands laboratoires américains.

In [ ]:
par_pays = analyses.opacite_par_pays(epoch_pays)
# Pays avec au moins 5 modèles récents : en dessous, une part n'a pas de sens
barres_par_pays(par_pays.loc[par_pays["modeles"] >= 5, "part_calcul_connu"],
                "Modèles publiés depuis 2024 dont la puissance de calcul est connue",
                "Part des modèles du pays", fichier=FIGURES / "rapport_6_opacite_pays.png")
plt.show()
display(par_pays.round(2))
analyses.opacite_leaders(epoch).round(2)

**Interprétation.** Depuis 2024, la puissance de calcul n'est connue que pour 27 % des modèles américains, contre 50 % des modèles chinois. L'écart vient des quatre grands laboratoires : OpenAI, Anthropic, Google DeepMind et xAI ne laissent connaître la puissance de calcul que de 13 % de leurs 113 modèles récents, contre 49 % pour les autres développeurs, alors qu'ils concentrent les modèles de pointe (12 % de leurs modèles, contre 2 %). Les modèles les plus avancés sont donc ceux qui manquent le plus dans la mesure du retard : le record américain, et donc l'écart avec la France, sont vraisemblablement **sous-estimés**. La France n'échappe pas à la tendance : la puissance de calcul n'est connue que pour 14 % de ses modèles récents (Mistral). Le Royaume-Uni, l'Allemagne et le Canada comptent moins de cinq modèles récents chacun : leurs parts ne sont pas interprétables.

## 4. De quoi parle la recherche française en IA ?

Corpus : résumés des 1 000 articles d'IA les plus cités depuis 2022 avec au moins un auteur affilié en France, comparés à ceux affiliés aux États-Unis. Nettoyage avec spaCy (lemmatisation, *stopwords*, mots génériques des résumés scientifiques), puis comparaison des vocabulaires par le **log-odds ratio à a priori de Dirichlet** (Monroe, Colaresi et Quinn, 2008) : un z-score supérieur à 2 signale un terme nettement plus fréquent dans un corpus que dans l'autre. Dans les nuages de mots, la taille d'un terme mesure la force de son association avec le pays.

In [ ]:
resumes = resumes.dropna(subset=["resume"])
resumes = resumes.loc[resumes["resume"].str.len() > 200]
resumes = resumes.loc[~resumes["titre"].fillna("").str.contains(r"\bquantum\b", case=False)]
resumes["pays"] = resumes["pays"].map({"fr": "France", "us": "États-Unis"})

nlp = spacy.load("en_core_web_sm")
resumes["tokens"] = nettoyer(resumes["resume"].tolist(), nlp)
freq = {p: frequences(resumes.loc[resumes["pays"] == p, "tokens"]) for p in ["France", "États-Unis"]}
comparaison_termes = termes_surrepresentes(freq["France"], freq["États-Unis"])
print(resumes.groupby("pays").size().to_dict())

In [ ]:
extremes = pd.concat([comparaison_termes.head(12), comparaison_termes.tail(12)])
barres_divergentes(extremes, "z", "Termes les plus distinctifs de chaque corpus",
                   "z-score (> 0 : associé à la France, < 0 : aux États-Unis)",
                   fichier=FIGURES / "rapport_4_termes.png")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
cote_fr = comparaison_termes.loc[comparaison_termes["z"] > 2].head(25)
cote_us = comparaison_termes.loc[comparaison_termes["z"] < -2].tail(25)
nuage(dict(zip(cote_fr["terme"], cote_fr["z"])), axes[0], "Associés à la France", COULEURS_PAYS["France"])
nuage(dict(zip(cote_us["terme"], -cote_us["z"])), axes[1], "Associés aux États-Unis", COULEURS_PAYS["États-Unis"])
fig.tight_layout(); fig.savefig(FIGURES / "rapport_5_nuages.png"); plt.show()

**Interprétation.** Les deux vocabulaires dessinent deux spécialisations. La recherche française en IA la plus citée est associée à la parole (*speech*, *speaker*), à la détection d'anomalies (*detection*, *anomaly*), à la santé (*patient*, *cancer*) et à l'IA explicable (*xai*, *explanation*) : des applications et des méthodes, plutôt que les grands modèles. La recherche américaine est dominée par les grands modèles de langage (*llm*, *prompt*, *reasoning*, *instruction*, *agent*). C'est cohérent avec la partie 2 : la France publie, mais pas dans le domaine où se joue aujourd'hui la course aux modèles. Limite : les 1 000 articles français les plus cités le sont beaucoup moins que les américains (médiane de 62 citations contre 508).

## Conclusion

**La France est en retard, et la transparence ne l'explique pas.** Son décrochage se situe entre la recherche, où elle tient un rang proche de son poids, et les modèles de pointe, qu'elle ne produit pas. Sa présence repose sur Mistral AI et sur des collaborations. Son retard en puissance de calcul sur les États-Unis oscille entre 1 et 4 ans depuis 2018, et atteint 3 ans en 2026.

L'opacité des leaders est bien réelle : la part des modèles dont la puissance de calcul est connue a été divisée par trois depuis 2021. Elle conduit plutôt à **sous-estimer** le retard : les quatre grands laboratoires américains ne laissent connaître la puissance de calcul que de 13 % de leurs modèles récents, contre 49 % pour les autres développeurs (partie 3.6). Enfin, la transparence se mesure mal : les modèles ouverts sont transparents techniquement mais opaques sur leurs données et leurs usages, et l'AI Act n'a pas, à ce stade, rendu ses signataires plus transparents.

**Limites principales** : Epoch ne retient que les modèles « notables » et mesure l'information connue, pas divulguée ; OpenAlex compte l'affiliation et non la nationalité ; un an seulement de recul après l'AI Act.

**Prolongements** : suivre les résumés de données d'entraînement publiés au titre de l'AI Act ; étendre la comparaison au reste de l'Europe ; mesurer la mobilité des chercheurs français vers l'industrie américaine.